# 📊 Notebook 07 — Projeção de KPI de Atingimento OLA
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Calcular o atingimento das metas anuais de OLA com base nos dados reais de 2025
e projetar o fechamento do ano, gerando o KPI de monitoramento da operação.

**Metas OLA Locaweb:**
- P2 (Alta): resolução ≤ 4h | meta: 36–39 violações/ano
- P3 (Média): resolução ≤ 12h | meta: 231–263 violações/ano

**Input:** `data/raw/LW-DATASET.xlsx`
**Output:** `outputs/kpi_atingimento.json`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import json, os
from datetime import date

print("✅ Imports ok")


In [ ]:
# ── Carregar dataset real ─────────────────────────────────────────────────────
raw = pd.read_excel('../data/raw/LW-DATASET.xlsx')
kpi = raw[raw['Entrou para KPI?']=='SIM'].copy()
kpi['Aberto'] = pd.to_datetime(kpi['Aberto'])
kpi['data']   = kpi['Aberto'].dt.normalize()
kpi['mes']    = kpi['Aberto'].dt.month
kpi['P']      = kpi['Prioridade'].map({'2 - Alta':'P2','3 - Média':'P3'})

print(f"Total incidentes KPI: {len(kpi)}")
print(f"Período: {kpi['data'].min().date()} a {kpi['data'].max().date()}")


In [ ]:
# ── Metas OLA ─────────────────────────────────────────────────────────────────
METAS = {
    'P2': {'min': 36, 'max': 39, 'ola_horas': 4},
    'P3': {'min': 231, 'max': 263, 'ola_horas': 12},
}

# Contagem de violações reais por mês e prioridade
viol_mes = kpi[kpi['Violou OLA?']=='SIM'].groupby(['mes','P']).size().unstack(fill_value=0)
viol_total = kpi[kpi['Violou OLA?']=='SIM'].groupby('P').size()

print("=== Violações reais 2025 ===")
print(viol_total.to_string())
print()
for p, meta in METAS.items():
    real = viol_total.get(p, 0)
    status = "✅ DENTRO" if meta['min'] <= real <= meta['max'] else "❌ FORA"
    print(f"{p}: {real} violações | meta: {meta['min']}-{meta['max']} | {status}")


In [ ]:
# ── Cálculo do KPI de atingimento ─────────────────────────────────────────────
def calcular_kpi(real, meta_min, meta_max):
    """
    KPI = quanto a operação está performando vs a meta.
    Se real <= meta_max: dentro da meta (atingimento >= 100%)
    Se real > meta_max: fora da meta (atingimento < 100%)
    Atingimento = meta_max / real * 100
    """
    if real == 0:
        return 100.0, 'atingida'
    pct = round(meta_max / real * 100, 1)
    if real <= meta_max:
        tendencia = 'dentro_da_meta'
    elif real <= meta_max * 1.15:
        tendencia = 'atencao'
    else:
        tendencia = 'critico'
    return pct, tendencia

kpi_result = {}
for p, meta in METAS.items():
    real = int(viol_total.get(p, 0))
    pct, tendencia = calcular_kpi(real, meta['min'], meta['max'])
    kpi_result[p] = {
        'violacoesAno': real,
        'metaMin': meta['min'],
        'metaMax': meta['max'],
        'pctAtingimento': pct,
        'tendencia': tendencia,
        'olaHoras': meta['ola_horas'],
    }
    print(f"{p}: {real} violações | {pct}% atingimento | {tendencia}")


In [ ]:
# ── Gráfico de atingimento mensal ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

for ax, p in zip(axes, ['P2', 'P3']):
    vals = [viol_mes.get(p, pd.Series()).get(m, 0) for m in range(1, 13)]
    meta_max = METAS[p]['max'] / 12  # meta mensal proporcional
    cores = ['#E8002D' if v > meta_max else '#34c759' for v in vals]
    ax.bar(meses, vals, color=cores, alpha=0.85)
    ax.axhline(meta_max, color='#ff9500', linestyle='--', linewidth=1.5,
               label=f'Meta mensal ({meta_max:.1f})')
    ax.set_title(f'Violações OLA {p} por mês — 2025')
    ax.set_xlabel('Mês'); ax.set_ylabel('Violações')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/kpi_violacoes_mensal.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico salvo")


In [ ]:
# ── Exportar kpi_atingimento.json ─────────────────────────────────────────────
output = {
    "modelo": "kpi_ola_atingimento",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "periodo": "2025-01-01 a 2025-12-31",
    "P2": kpi_result['P2'],
    "P3": kpi_result['P3'],
    "por_mes": {
        p: {str(m): int(viol_mes.get(p, pd.Series()).get(m, 0)) for m in range(1,13)}
        for p in ['P2', 'P3']
    },
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/kpi_atingimento.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ kpi_atingimento.json exportado")
print(json.dumps(output, ensure_ascii=False, indent=2))
